# Qualidade do ar em Sao Paulo

Este notebook consulta a API Open-Meteo para obter dados de qualidade do ar em Sao Paulo. Em seguida, ele calcula uma media para cada poluente por dia e salva o resultado em um arquivo CSV.

O periodo consultado vai de **06/09/2024** ate **06/09/2026**, totalizando dois anos de dados diarios.

## Bibliotecas

Execute a proxima celula apenas se as bibliotecas ainda nao estiverem instaladas. O `requests_cache` cria um arquivo `.cache.sqlite` para guardar respostas por uma hora e evitar requisicoes repetidas.

In [ ]:
%pip install openmeteo-requests pandas requests-cache retry-requests

## Codigo completo

A API disponibiliza as medicoes em intervalos horarios. Para termos somente uma linha por dia, o codigo calcula a media diaria de cada poluente. A coluna `date` e salva sem horario.

In [ ]:
from pathlib import Path

import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
PARAMS = {
    "latitude": -23.5489,
    "longitude": -46.6388,
    "start_date": "2024-09-06",
    "end_date": "2026-09-06",
    "hourly": [
        "pm10", "pm2_5", "carbon_monoxide", "carbon_dioxide",
        "nitrogen_dioxide", "sulphur_dioxide", "ozone",
    ],
}
OUTPUT_CSV = Path("dados_saida") / "qualidade_do_ar_por_dia.csv"

responses = openmeteo.weather_api(URL, params=PARAMS)
response = responses[0]
hourly = response.Hourly()

# A ordem dos indices deve acompanhar PARAMS["hourly"].
hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ),
    "pm10": hourly.Variables(0).ValuesAsNumpy(),
    "pm2_5": hourly.Variables(1).ValuesAsNumpy(),
    "carbon_monoxide": hourly.Variables(2).ValuesAsNumpy(),
    "carbon_dioxide": hourly.Variables(3).ValuesAsNumpy(),
    "nitrogen_dioxide": hourly.Variables(4).ValuesAsNumpy(),
    "sulphur_dioxide": hourly.Variables(5).ValuesAsNumpy(),
    "ozone": hourly.Variables(6).ValuesAsNumpy(),
}

hourly_dataframe = pd.DataFrame(hourly_data)
daily_dataframe = (
    hourly_dataframe.set_index("date").resample("D").mean().reset_index()
)
daily_dataframe["date"] = daily_dataframe["date"].dt.strftime("%Y-%m-%d")

OUTPUT_CSV.parent.mkdir(exist_ok=True)
daily_dataframe.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
daily_dataframe

## Resultado

Ao executar a celula de codigo, o DataFrame diario aparece abaixo dela e o arquivo `dados_saida/qualidade_do_ar_por_dia.csv` e criado. Cada linha representa um dia e cada coluna representa a media diaria de um poluente.